# **Hate Speech Classification**

Tujuan project: Membangun model klasifikasi biner menggunakan TF-IDF dan Support Vector Machine yang mampu mengklasifikasikan teks berbahasa Indonesia menjadi kelas hate speech dan non-hate speech.


## **Import Library**

In [5]:
!pip install nltk
!pip install nlpaug
!pip install Sastrawi
!pip install scikit-learn
!pip install swifter

Access is denied.
Access is denied.
Access is denied.
Access is denied.
Access is denied.


In [9]:
import pandas as pd
import numpy as np
import string, re

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

import matplotlib.pyplot as plt

import swifter
import csv

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## **Load Dataset**

In [10]:
# Membaca dataset hate speech
df = pd.read_csv('../dataset/processed/data_eda.csv', encoding='latin-1')

In [11]:
# Menampilkan informasi struktur dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13023 entries, 0 to 13022
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Tweet   13023 non-null  object
 1   HS      13023 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 203.6+ KB


## **Preprocessing Data**

CLEANING

In [12]:
# Cleaning text
def cleaning_text(text):
    # Decode representasi byte UTF-8 seperti \xf0\x9f\x98\x82
    def decode_utf8_escape(match):
        hex_values = re.findall(r'\\x([0-9a-fA-F]{2})', match.group())
        byte_data = bytes(int(value, 16) for value in hex_values)

        return byte_data.decode('utf-8', errors='replace')

    text = re.sub(r'(?:\\x[0-9a-fA-F]{2})+', decode_utf8_escape, text)

    # Mengganti newline, tab, dan carriage return
    text = re.sub(r'\\[nrt]', ' ', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)

    # Menghapus URL, hastag, RT
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'#([A-Za-z0-9_]+)', r'\1', text)
    text = re.sub(r'\bRT\b', ' ', text, flags=re.IGNORECASE)

    # Menghapus angka
    text = re.sub(r'[0-9]+', '', text)

    # Menghapus karakter selain huruf dan spasi
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    text = text.replace('_', ' ')

    # Menyeragamkan placeholder USER
    text = re.sub(r'\bUSER\b', 'USER', text, flags=re.IGNORECASE)
    text = re.sub(r'(?:\bUSER\b\s*){2,}', 'USER ', text)

    # Merapikan spasi
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['cleanpunct'] = df['Tweet'].apply(cleaning_text)
cleanpunct = pd.DataFrame(df['cleanpunct'])
cleanpunct

,cleanpunct
0,disaat semua cowok berusaha melacak perhatian ...
1,USER siapa yang telat ngasih tau elu edan sara...
2,Kadang aku berfikir kenapa aku tetap percaya p...
3,USER AKU ITU AKU KU TAU MATAMU SIPIT TAPI DILI...
4,USER Kaum cebong kapir udah keliatan dongoknya...
...,...
13018,USER jangan asal ngomong ndasmu congor lu yg s...
13019,USER Kasur mana enak kunyuk
13020,USER Hati hati bisu g lagi bosan huft
13021,USER Bom yang real mudah terdeteksi bom yang t...


CASE FOLDING

In [13]:
# Case folding
df['lowercase'] = df['cleanpunct'].str.lower()

casefolding = pd.DataFrame(df['lowercase'])
casefolding

,lowercase
0,disaat semua cowok berusaha melacak perhatian ...
1,user siapa yang telat ngasih tau elu edan sara...
2,kadang aku berfikir kenapa aku tetap percaya p...
3,user aku itu aku ku tau matamu sipit tapi dili...
4,user kaum cebong kapir udah keliatan dongoknya...
...,...
13018,user jangan asal ngomong ndasmu congor lu yg s...
13019,user kasur mana enak kunyuk
13020,user hati hati bisu g lagi bosan huft
13021,user bom yang real mudah terdeteksi bom yang t...


TOKENIZATION

In [14]:
# Tokenizing text
def tokenizingText(text):
    tokens = word_tokenize(text)
    return tokens

df['tokens'] = df['lowercase'].apply(tokenizingText)
tokenize = pd.DataFrame(df['tokens'])
tokenize

,tokens
0,"[disaat, semua, cowok, berusaha, melacak, perh..."
1,"[user, siapa, yang, telat, ngasih, tau, elu, e..."
2,"[kadang, aku, berfikir, kenapa, aku, tetap, pe..."
3,"[user, aku, itu, aku, ku, tau, matamu, sipit, ..."
4,"[user, kaum, cebong, kapir, udah, keliatan, do..."
...,...
13018,"[user, jangan, asal, ngomong, ndasmu, congor, ..."
13019,"[user, kasur, mana, enak, kunyuk]"
13020,"[user, hati, hati, bisu, g, lagi, bosan, huft]"
13021,"[user, bom, yang, real, mudah, terdeteksi, bom..."


NORMALIZATION

In [15]:
# Normalisasi menggunakan kamus
alay_dict = pd.read_csv("../dataset/raw/new_kamusalay.csv", encoding='latin-1', header=None)
alay_dict.columns = ["original", "replacement"]

# Memeriksa isi dan nama kolom kamus
print("Shape: ", alay_dict.shape)
alay_dict.head(10)

Shape:  (15175, 2)


,original,replacement
0,anakjakartaasikasik,anak jakarta asyik asyik
1,pakcikdahtua,pak cik sudah tua
2,pakcikmudalagi,pak cik muda lagi
3,t3tapjokowi,tetap jokowi
4,3x,tiga kali
5,aamiin,amin
6,aamiinn,amin
7,aamin,amin
8,aammiin,amin
9,abis,habis


In [16]:
# Normalisasi text
alay_dict_map = dict(zip(alay_dict['original'], alay_dict['replacement']))

def normalizationText(tokens):
    if isinstance(tokens, str):
        tokens = tokens.split()

    normalized_tokens = []

    for word in tokens:
        replacement = alay_dict_map.get(word, word)

        normalized_tokens.extend(replacement.split())

    return normalized_tokens

df["normalization"] = df["tokens"].apply(normalizationText)
normalisasi = pd.DataFrame(df['normalization'])
normalisasi

,normalization
0,"[di, saat, semua, cowok, berusaha, melacak, pe..."
1,"[pengguna, siapa, yang, telat, memberi, tau, k..."
2,"[kadang, aku, berpikir, kenapa, aku, tetap, pe..."
3,"[pengguna, aku, itu, aku, aku, tau, matamu, si..."
4,"[pengguna, kaum, cebong, kafir, sudah, kelihat..."
...,...
13018,"[pengguna, jangan, asal, berbicara, ndasmu, co..."
13019,"[pengguna, kasur, mana, enak, kunyuk]"
13020,"[pengguna, hati, hati, bisu, tidak, lagi, bosa..."
13021,"[pengguna, bom, yang, real, mudah, terdeteksi,..."


STOPWORD REMOVAL

In [17]:
# Stopword bahasa Indonesia
factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

def stopwordRemoval(tokens):
    return [word for word in tokens if word not in stopwords]

df["stopword"] = df["normalization"].apply(stopwordRemoval)
stopword = pd.DataFrame(df['stopword'])
stopword

,stopword
0,"[semua, cowok, berusaha, melacak, perhatian, g..."
1,"[pengguna, siapa, telat, memberi, tau, kamu, e..."
2,"[kadang, aku, berpikir, aku, tetap, percaya, t..."
3,"[pengguna, aku, aku, aku, tau, matamu, sipit, ..."
4,"[pengguna, kaum, cebong, kafir, kelihatan, don..."
...,...
13018,"[pengguna, jangan, asal, berbicara, ndasmu, co..."
13019,"[pengguna, kasur, mana, enak, kunyuk]"
13020,"[pengguna, hati, hati, bisu, bosan, duh]"
13021,"[pengguna, bom, real, mudah, terdeteksi, bom, ..."


STEMMING

In [18]:
# Stemming Text
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemmingText(tokens):
   stems = [stemmer.stem(word) for word in tokens]
   return ' '.join(stems)

df["stemming"] = df["stopword"].swifter.apply(stemmingText)
stemming = pd.DataFrame(df['stemming'])
stemming

Pandas Apply:   0%|          | 0/13023 [00:00<?, ?it/s]

,stemming
0,semua cowok usaha lacak perhati gue kamu lanta...
1,guna siapa telat beri tau kamu edan sarap gue ...
2,kadang aku pikir aku tetap percaya tuhan padah...
3,guna aku aku aku tau mata sipit lihat mana aku
4,guna kaum cebong kafir lihat dongok awal tamba...
...,...
13018,guna jangan asal bicara ndasmu congor kamu kat...
13019,guna kasur mana enak kunyuk
13020,guna hati hati bisu bosan duh
13021,guna bom real mudah deteksi bom kubur suatu le...


In [20]:
df.to_csv('../dataset/processed/data_preprocessing.csv', index=False)
print("File berhasil disimpan")

File berhasil disimpan
